In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Task 6 Commit 1: Prepare Tableau Data for Dashboard 1 & Dashboard 2
# feat(task6): export mortgage overview and model evaluation datasets
#
# Coursework Requirement:
# Prepare Tableau-ready datasets for visual analytics.
# Dashboard 1 : Mortgage Application Overview
# Dashboard 2 : Machine Learning Model Performance
# ──────────────────────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import pandas as pd
import builtins

os.environ["JAVA_HOME"]="/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PYSPARK_PYTHON"]=sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("7006SCN_HMDA_Task6")
    .master("local[*]")
    .config("spark.driver.memory","10g")
    .config("spark.sql.shuffle.partitions","200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# ------------------------------------------------------------------------------
# Project Paths
# ------------------------------------------------------------------------------

BASE="/content/drive/MyDrive/7006SCN"

PROCESSED=f"{BASE}/data/processed"

OUTPUT=f"{BASE}/outputs"

print("="*80)
print("TASK 6 : TABLEAU DATA PREPARATION")
print("="*80)

# ------------------------------------------------------------------------------
# Load Clean HMDA Dataset
# ------------------------------------------------------------------------------

df=spark.read.parquet(
    f"{PROCESSED}/hmda_clean.parquet"
)

print(f"Rows    : {df.count():,}")
print(f"Columns : {len(df.columns)}")

# ==============================================================================
# DASHBOARD 1
# Mortgage Application Overview
# ==============================================================================

print("\nPreparing Dashboard 1...")

# ------------------------------------------------------------------------------
# KPI Summary
# ------------------------------------------------------------------------------

total=df.count()

approved=df.filter(
    col("loan_approval")==1
).count()

rejected=df.filter(
    col("loan_approval")==0
).count()

approval_rate = builtins.round(
    approved / total * 100,
    2
)

kpi=pd.DataFrame({

    "Metric":[
        "Total Applications",
        "Approved Loans",
        "Rejected Loans",
        "Approval Rate (%)"
    ],

    "Value":[
        total,
        approved,
        rejected,
        approval_rate
    ]

})

kpi.to_csv(
    f"{OUTPUT}/t6_dashboard1_kpi.csv",
    index=False
)

# ------------------------------------------------------------------------------
# Missing Values
# ------------------------------------------------------------------------------

missing=[]

for c in df.columns:

    missing.append({

        "Column":c,

        "Missing Values":df.filter(
            col(c).isNull()
        ).count()

    })

missing_df=pd.DataFrame(missing)

missing_df.to_csv(

    f"{OUTPUT}/t6_dashboard1_missing.csv",

    index=False

)

# ------------------------------------------------------------------------------
# Loan Approval Distribution
# ------------------------------------------------------------------------------

approval=(
    df.groupBy("loan_approval")
      .count()
      .toPandas()
)

approval.to_csv(

    f"{OUTPUT}/t6_dashboard1_approval.csv",

    index=False

)

# ------------------------------------------------------------------------------
# Loan Purpose Distribution
# ------------------------------------------------------------------------------

if "loan_purpose" in df.columns:

    purpose=(
        df.groupBy("loan_purpose")
          .count()
          .toPandas()
    )

    purpose.to_csv(

        f"{OUTPUT}/t6_dashboard1_loanpurpose.csv",

        index=False

    )

# ------------------------------------------------------------------------------
# Race Distribution
# ------------------------------------------------------------------------------

if "applicant_race_1" in df.columns:

    race=(
        df.groupBy("applicant_race_1")
          .count()
          .toPandas()
    )

    race.to_csv(

        f"{OUTPUT}/t6_dashboard1_race.csv",

        index=False

    )

# ------------------------------------------------------------------------------
# Ethnicity Distribution
# ------------------------------------------------------------------------------

if "applicant_ethnicity_1" in df.columns:

    ethnicity=(
        df.groupBy("applicant_ethnicity_1")
          .count()
          .toPandas()
    )

    ethnicity.to_csv(

        f"{OUTPUT}/t6_dashboard1_ethnicity.csv",

        index=False

    )

print("Dashboard 1 datasets exported ✓")

# ==============================================================================
# DASHBOARD 2
# Machine Learning Performance
# ==============================================================================

print("\nPreparing Dashboard 2...")

metrics=pd.read_csv(
    f"{OUTPUT}/task5_metrics.csv"
)

metrics.to_csv(

    f"{OUTPUT}/t6_dashboard2_model_metrics.csv",

    index=False

)

print("Model metrics loaded ✓")

# ------------------------------------------------------------------------------
# SHAP Feature Importance
# ------------------------------------------------------------------------------

shap_df=pd.read_csv(

    f"{OUTPUT}/task5_shap.csv"

)

shap_df.to_csv(

    f"{OUTPUT}/t6_dashboard2_shap.csv",

    index=False

)

print("SHAP importance loaded ✓")

# ------------------------------------------------------------------------------
# Prediction Time Dataset
# ------------------------------------------------------------------------------

prediction=metrics[

    [

        "Model",

        "Prediction Time (s)"

    ]

]

prediction.to_csv(

    f"{OUTPUT}/t6_dashboard2_prediction_time.csv",

    index=False

)

# ------------------------------------------------------------------------------
# ROC Comparison
# ------------------------------------------------------------------------------

roc=metrics[

    [

        "Model",

        "ROC-AUC"

    ]

]

roc.to_csv(

    f"{OUTPUT}/t6_dashboard2_roc.csv",

    index=False

)

print("Dashboard 2 datasets exported ✓")

print()

print("="*80)
print("COMMIT 1 COMPLETED")
print("="*80)